In [ ]:
import pandas as pd
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from collections import Counter
from concurrent.futures import ThreadPoolExecutor


In [2]:
data1 = pd.read_csv("tfidf_try.csv")

In [3]:
data = data1[:2000]

In [ ]:
def safe_eval(x):
    """Safely convert stringified list to Python list, handle NaN."""
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return []

#  parallel safe_eval across all 3 columns at once 
cols = ["MeSH Terms (MH)", "Other Terms (OT)", "Registry Numbers (RN)"]
with ThreadPoolExecutor() as executor:
    futures = {col: executor.submit(lambda s: s.apply(safe_eval), data[col])
               for col in cols}
    for col, fut in futures.items():
        data[col] = fut.result()

/tmp/ipykernel_888979/597294463.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[col] = data[col].apply(safe_eval)
/tmp/ipykernel_888979/597294463.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["combined"] = data.apply(


In [ ]:
# vectorized list combination (replaces slow axis=1 apply)
data["combined"] = (
    data["MeSH Terms (MH)"]
    + data["Other Terms (OT)"]
    + data["Registry Numbers (RN)"]
)

In [ ]:
## now im thinking whether i should keep row or create one big document, will go with 2 for now:
# Option 1: Keep row-wise documents (good for TF-IDF per document)
# Each row = one document
#corpus = [" ".join(tokens) for tokens in data["combined"]]

In [ ]:
# parallel group document building
group_masks = {
    "Elsevier=1":      data["Elsevier"] == 1, # Group 1: Elsevier == 1 ######### ONLY MESH TERMS, OT, RN
    "Elsevier=0":      data["Elsevier"] == 0, # Group 2: Elsevier == 0
######################
    "Elsevier+OA":     (data["Elsevier"] == 1) & (data["OA-noncomm"] == 1), # Group 3: Elsevier + OA  ############ MESH + FULL_TEXT
    "Elsevier NON OA": (data["Elsevier"] == 1) & (data["OA-noncomm"] == 0), # Group 4: Elsevier_NON_OA == 1
######################
    "OA-noncomm=1":    data["OA-noncomm"] == 1, # Group 5: OA == 1############### ONLY MESH TERMS, OT, RN
    "OA!=1":           data["OA-noncomm"] == 0, # Group 6: OA != 1

}


In [ ]:
def build_group_doc(mask):
    tokens = data.loc[mask, "combined"].sum()
    return " ".join(tokens)

with ThreadPoolExecutor() as executor:
    futures = {label: executor.submit(build_group_doc, mask)
               for label, mask in group_masks.items()}
    group_docs = {label: fut.result() for label, fut in futures.items()}

elsevier_doc    = group_docs["Elsevier=1"]
elsevier0_doc   = group_docs["Elsevier=0"]
elsevier_oa_doc = group_docs["Elsevier+OA"]
elsevier2_doc   = group_docs["Elsevier NON OA"]
oa_doc          = group_docs["OA-noncomm=1"]
oa2_doc         = group_docs["OA!=1"]

In [ ]:
#3. Frequency counts over the full flattened corpus (for sanity check)
all_tokens = sum(data["combined"], [])
word_counts = Counter(all_tokens)
freq_df = pd.DataFrame(word_counts.most_common(), columns=["term", "count"])
print("Top 20 terms across full corpus:")
print(freq_df.head(20))

                       term  count
0                    Humans    867
1                   Animals    825
2                      Mice    377
3                      Male    311
4                    Female    294
5         Models, Molecular    184
6           Protein Binding    181
7    Crystallography, X-Ray    166
8   Molecular Sequence Data    160
9                      Rats    144
10         Cell Line, Tumor    142
11      Amino Acid Sequence    133
12            Binding Sites    126
13                Cell Line    123
14          Cells, Cultured    105
15           RNA, Messenger    102
16                  Ligands    100
17                    Adult     98
18    Transcription Factors     96
19                 Mutation     91


In [ ]:
#4. TF-IDF over grouped corpus
corpus = [elsevier_doc, oa_doc, elsevier2_doc, oa2_doc, elsevier_oa_doc]
labels = ["Elsevier", "OA-noncomm", "Elsevier NON OA", "OA != 1", "Elsevier + OA"]

vectorizer = TfidfVectorizer(
    tokenizer=lambda x: x.split(),
    preprocessor=lambda x: x,
    token_pattern=None
)

X = vectorizer.fit_transform(corpus)

tfidf_df = pd.DataFrame(
    X.toarray(),
    index=labels,
    columns=vectorizer.get_feature_names_out()
)

print(f"\nTF-IDF matrix shape: {tfidf_df.shape}  ({len(labels)} groups × vocabulary size)")

(3, 11126)


,&,(1-42,(3R,(ADCYAP1R1),(ADP-Ribose,(ADP-Ribose),(ADSCs),(APAR),(Activation-Induced,(CAMs),...,yeast,yellow,yellows,yield,zalihae,zebrafish,zeste,zeta,zinc,Ⅰ
Elsevier,0.166826,0.001885,0.001214,0.000000,0.001214,0.001214,0.000000,0.000000,0.002427,0.001214,...,0.000000,0.000000,0.000000,0.001214,0.002427,0.000000,0.001214,0.000943,0.001214,0.000000
OA-noncomm,0.137887,0.001053,0.001355,0.001782,0.000000,0.000000,0.001782,0.001782,0.000000,0.000000,...,0.001782,0.001782,0.001782,0.000000,0.000000,0.001782,0.000000,0.001053,0.000000,0.001782
Elsevier NON OA,0.170895,0.002022,0.000000,0.000000,0.001302,0.001302,0.000000,0.000000,0.002604,0.001302,...,0.000000,0.000000,0.000000,0.001302,0.002604,0.000000,0.001302,0.001011,0.001302,0.000000


In [10]:
top_n = 10

# Loop through each group/document and print terms
for label in tfidf_df.index:
    print(f"\nTop {top_n} terms for '{label}':")
    # Get TF-IDF scores for this document, sort descending, take top N
    top_terms = tfidf_df.loc[label].sort_values(ascending=False).head(top_n)
    for term, score in top_terms.items():
        print(f"  {term}: {score:.4f}")


Top 10 terms for 'Elsevier':
  Protein: 0.4298
  Proteins: 0.3789
  protein,: 0.2828
  Cell: 0.2111
  Humans: 0.2036
  Animals: 0.1904
  Sequence: 0.1706
  &: 0.1668
  effects: 0.1621
  Molecular: 0.1583

Top 10 terms for 'OA-noncomm':
  Proteins: 0.4200
  Protein: 0.3652
  protein,: 0.3326
  Humans: 0.2389
  Cell: 0.2147
  Animals: 0.2021
  human: 0.1684
  effects: 0.1558
  &: 0.1379
  Gene: 0.1274

Top 10 terms for 'Elsevier NON OA':
  Protein: 0.4186
  Proteins: 0.3752
  protein,: 0.2771
  Cell: 0.2174
  Humans: 0.2033
  Animals: 0.1931
  Sequence: 0.1729
  &: 0.1709
  effects: 0.1689
  Molecular: 0.1588
